In [24]:
# Basic setup
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score


In [25]:
# Load your CSV file
df = pd.read_csv("/content/5.loan_data.csv")

# Check shape and preview
print("Data shape:", df.shape)
df.head()

Data shape: (1000, 6)


,credit_score,loan_amount,income,employment_length,debt_to_income,loan_status
0,720,39012,54904,5,21.515381,1
1,779,12226,64834,8,38.682835,0
2,667,47927,32332,14,38.574933,1
3,380,23031,29296,13,10.766097,1
4,730,19984,93449,7,11.061158,1


In [26]:
target = "loan_status"  # change if your column name is different
y = (df[target] == "Default").astype(int)  # 1 = risky, 0 = safe
X = df.drop(columns=[target])


In [27]:
# Separate numeric & categorical columns
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(exclude=np.number).columns

# Pipelines for preprocessing
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])


In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [29]:
y.value_counts()



,count
loan_status,
0,1000


In [30]:

# Create a sample dataset with both approved (0) and default (1) loans
import pandas as pd
import numpy as np

np.random.seed(42)

df = pd.DataFrame({
    'income': np.random.randint(20000, 150000, 1000),
    'loan_amount': np.random.randint(5000, 60000, 1000),
    'credit_score': np.random.randint(300, 850, 1000),
    'loan_term': np.random.choice([12, 24, 36, 48, 60], 1000),
    'age': np.random.randint(21, 65, 1000),
    'loan_status': np.random.choice([0, 1], size=1000, p=[0.8, 0.2])  # 0=safe, 1=risky
})

print("✅ Sample data created with both classes")
df['loan_status'].value_counts()


✅ Sample data created with both classes


,count
loan_status,
0,810
1,190


In [31]:
# Target variable
target = 'loan_status'

# Separate features (X) and target (y)
X = df.drop(columns=[target])
y = df[target]


In [32]:
from sklearn.model_selection import train_test_split

# Stratify ensures both 0 and 1 appear in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training data shape:", X_train.shape)
print("Test data shape:", X_test.shape)
print("\nClass distribution in training:")
print(y_train.value_counts(normalize=True))


Training data shape: (800, 5)
Test data shape: (200, 5)

Class distribution in training:
loan_status
0    0.81
1    0.19
Name: proportion, dtype: float64


In [33]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

import numpy as np

# Separate numeric and categorical columns
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

# Pipelines for preprocessing
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine them together
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])


In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline # <-- This line fixes the error

# The fix: Adding class_weight='balanced'
model = Pipeline([
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

# Train the model
model.fit(X_train, y_train)

print("✅ Model training complete with balanced weights")

✅ Model training complete with balanced weights


In [35]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Evaluation metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.475
Precision: 0.15463917525773196
Recall: 0.39473684210526316
F1 Score: 0.2222222222222222
ROC AUC: 0.4582521117608837

Confusion Matrix:
 [[80 82]
 [23 15]]

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.49      0.60       162
           1       0.15      0.39      0.22        38

    accuracy                           0.47       200
   macro avg       0.47      0.44      0.41       200
weighted avg       0.66      0.47      0.53       200

